# Series, DataFrame, read_csv / read_json, Profiling, Chunks

## What you'll practice

pandas arrives *after* plain Python on purpose: now you know what a group-by or a join costs, pandas lets you express them in one line and run them in C.
Rule #1: think in **columns**, not rows. Rule #2: control dtypes at read time. All exercises use the real datasets.

> Function and class names in **Implement** are what the tests look for. Run all tests for this notebook with `pytest tests/pandas/test_09_01_dataframe_basics_io.py -v` from the lab root. Untouched exercises show as *skipped*.
>
> Solutions: `solutions/09_pandas/01_dataframe_basics_io.ipynb`. Don't open it until you've tried.

### Setup — run this first (sample data and imports, not a solution)

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
pd.set_option("display.width", 140)
LAB = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").is_dir() and (p / "labkit").is_dir())   # lab root (works from any folder)
DATA = LAB / "datasets"
orders_raw = pd.read_csv(DATA / "csv" / "orders.csv", dtype=str, keep_default_na=False)
print(orders_raw.shape); orders_raw.head()

## Exercise 1 — 🟢 Easy · Load with explicit dtypes

**Implement:** `load_orders(path) -> pd.DataFrame`

Read `orders.csv` so that ids stay strings (`order_id`, `customer_id`, `product_id`), `quantity` and `unit_price` are numeric (invalid → NaN via
`pd.to_numeric(errors="coerce")`), and `order_ts`/`updated_at` are tz-aware UTC datetimes (invalid → NaT; use `pd.to_datetime(..., utc=True, format="ISO8601", errors="coerce")`).

**Input:** `path`

**Expected output:** `pd.DataFrame`

**Constraints:** pandas / numpy; prefer vectorized column operations over row loops (`iterrows`, `apply(axis=1)`).

**Example:** the tests are executable examples — read `test_load_orders` in `tests/pandas/test_09_01_dataframe_basics_io.py`.

**Check:** `pytest tests/pandas/test_09_01_dataframe_basics_io.py -k test_load_orders`

**YOUR CODE ↓**

## Exercise 2 — 🟢 Easy · Series statistics

**Implement:** `series_stats(s: pd.Series) -> dict`

Return `{"count" (non-null), "nulls", "mean", "median", "p95", "min", "max"}` with floats rounded to 2. Empty/all-null series → count 0 and `None` for the numeric stats.

**Input:** `s: pd.Series`

**Expected output:** `dict`

**Constraints:** pandas / numpy; prefer vectorized column operations over row loops (`iterrows`, `apply(axis=1)`).

**Example:** the tests are executable examples — read `test_series_stats` in `tests/pandas/test_09_01_dataframe_basics_io.py`.

**Check:** `pytest tests/pandas/test_09_01_dataframe_basics_io.py -k test_series_stats`

**YOUR CODE ↓**

## Exercise 3 — 🟡 Medium · Flatten nested JSON with json_normalize

**Implement:** `load_order_items(path) -> pd.DataFrame`

Load `datasets/json/orders_nested.json` into **one row per item** using `pd.json_normalize(data, record_path="items", meta=[...], errors="ignore")`. Output columns exactly:
`order_id, created_at, status, customer_id, country, payment_status, product_id, name, category, qty, price, line_total` (line_total = qty × price). Orders with no items vanish (that's OK here);
orders without `payment` get NaN payment_status.

**Input:** `path`

**Expected output:** `pd.DataFrame`

**Constraints:** pandas / numpy; prefer vectorized column operations over row loops (`iterrows`, `apply(axis=1)`).

**Example:** the tests are executable examples — read `test_load_order_items` in `tests/pandas/test_09_01_dataframe_basics_io.py`.

**Check:** `pytest tests/pandas/test_09_01_dataframe_basics_io.py -k test_load_order_items`

**YOUR CODE ↓**

## Exercise 4 — 🟠 Hard · Profile a DataFrame

**Implement:** `profile(df: pd.DataFrame) -> pd.DataFrame`

Return a profile indexed by column name with columns `dtype` (str), `non_null`, `nulls`, `null_pct` (0–100, 2 dp), `distinct` (excluding nulls), `empty_strings` (count of `""`/whitespace
in string-like columns, 0 otherwise), `example` (first non-null value as str, or `None`). Build it **column-wise** with vectorized operations, not by iterating rows.

**Input:** `df: pd.DataFrame`

**Expected output:** `pd.DataFrame`

**Constraints:** pandas / numpy; prefer vectorized column operations over row loops (`iterrows`, `apply(axis=1)`).

**Example:** the tests are executable examples — read `test_profile` in `tests/pandas/test_09_01_dataframe_basics_io.py`.

**Check:** `pytest tests/pandas/test_09_01_dataframe_basics_io.py -k test_profile`

**YOUR CODE ↓**

## Challenge — 🔴 Very Hard · Process a file in chunks

**Implement:** `revenue_by_country_chunked(path, chunksize: int = 1000) -> pd.Series`

Pretend `orders.csv` doesn't fit in memory. Use `pd.read_csv(..., chunksize=...)` to stream it; per chunk keep rows with numeric quantity ≥ 1 and numeric unit_price ≥ 0,
dedupe within the chunk by exact row, compute revenue per country, and **combine partial aggregates** across chunks (sum of sums). Return a Series named `revenue`, index named `country`,
sorted descending, rounded to 2. It must equal the non-chunked computation when there are no cross-chunk duplicates — then compare `chunksize=700` with one giant chunk on the
real (shuffled) file and explain the difference.

**Input:** `path, chunksize: int = 1000`

**Expected output:** `pd.Series`

**Constraints:** pandas / numpy; prefer vectorized column operations over row loops (`iterrows`, `apply(axis=1)`).

**Example:** the tests are executable examples — read `test_revenue_by_country_chunked, test_chunked_matches_full` in `tests/pandas/test_09_01_dataframe_basics_io.py`.

**Check:** `pytest tests/pandas/test_09_01_dataframe_basics_io.py -k "test_revenue_by_country_chunked or test_chunked_matches_full"`

**YOUR CODE ↓**